In [ ]:
!git clone https://github.com/docty/image-creation.git

In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(),'image-creation/utils'))

In [ ]:
!pip install -r {os.getcwd()}/image-creation/requirements.txt -q

In [ ]:
import torch
from diffusers import StableDiffusionImg2ImgPipeline, StableDiffusionPipeline
from diffusers import EulerAncestralDiscreteScheduler, DPMSolverMultistepScheduler
from diffusers.utils import load_image
import gradio as gr

device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"

pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16).to(device)

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

#pipe.vae = AutoencoderKL.from_pretrained('stabilityai/sd-vae-ft-mse', torch_dtype=torch.float16).to("cuda")

pipe.enable_attention_slicing()

In [ ]:
from prompt_loader import load_prompts

PROMPT_OPTIONS = load_prompts(os.path.join(os.getcwd(),'image-creation/utils/prompt_options.json'))

negative_prompt = "3D cartoon, illustration, CGI, blurry, smooth surfaces, fantasy, bright daylight,  abstract, low detail, low resolution"

STRENGTH = 0.75
GUIDANCE_SCALE = 7.5

generator = torch.Generator("cuda").manual_seed(0)

In [ ]:
examples = [
    [
        'https://raw.githubusercontent.com/docty/image-creation/refs/heads/main/assets/dent_1.jpg',
        'Dent',
    ],
    
    [
        'https://raw.githubusercontent.com/docty/image-creation/refs/heads/main/assets/bend_3.jpg',
        'Bend',
    ],
    [
        'https://raw.githubusercontent.com/docty/image-creation/refs/heads/main/assets/coating_1.jpg',
        'Chip Coating',
    ],
     [
        'https://raw.githubusercontent.com/docty/image-creation/refs/heads/main/assets/coating_7.jpg',
        'Stain Coating',
    ],
    [ 
        'https://raw.githubusercontent.com/docty/image-creation/refs/heads/main/assets/corrosion_1.jpg',
        'Corrosion',
    ],
    [
        'https://raw.githubusercontent.com/docty/image-creation/refs/heads/main/assets/debris_1.jpg',
        'Debris',
    ],
    [
        'https://raw.githubusercontent.com/docty/image-creation/refs/heads/main/assets/marine_growth_1.jpg',
        'Marine Growth',
    ],
    [
        'https://raw.githubusercontent.com/docty/image-creation/refs/heads/main/assets/crack_4.jpg',
        'Crack',
    ],
    
     
]

In [ ]:
def generate_image(init_image, prompt_choice, inputText):
    init_image = init_image.convert("RGB").resize((512, 768))
    prompt = PROMPT_OPTIONS[prompt_choice]
    prompt = inputText
     
    result = pipe(
        prompt=prompt,
        image=init_image,
        strength=STRENGTH,
        guidance_scale=GUIDANCE_SCALE,
        negative_prompt=negative_prompt
    ).images[0]

    return result

def update_input_text(prompt_choice):
        return PROMPT_OPTIONS.get(prompt_choice, "")
    
with gr.Blocks(css="""
    .gr-block { padding: 1rem !important; }
    .gr-button { width: 100%; font-weight: bold; }
    .gr-image { border-radius: 12px; }
     body {background-color: black;}
""") as demo:

    gr.Markdown("""
        <h1 style='text-align: center; color: #333;'> Image Generator</h1>
        <p style='text-align: center; max-width: 700px; margin: auto; color: #555;'>
            Upload an initial image and choose a corrosion style prompt to generate an image or choose an Example from below.
        </p>
    """)

    
    with gr.Row(equal_height=True):
        prompt_choice = gr.Dropdown(
            choices=list(PROMPT_OPTIONS.keys()),
            label="Select Anomalies Type",
            value=None
        )
        generate_btn = gr.Button("Generate Image")

    inputText = gr.Textbox(label='Enter Prompt')
    with gr.Row():
        with gr.Column(scale=1):
            init_image = gr.Image(
                label="Upload Your Image",
                type="pil",
                
            )
        with gr.Column(scale=1):
            output_image = gr.Image(label="Generated Image", height=300)

    generate_btn.click(
        fn=generate_image,
        inputs=[init_image, prompt_choice, inputText],
        outputs=output_image
    )

    
    prompt_choice.change(
        fn=update_input_text, 
        inputs=prompt_choice, 
        outputs=inputText
    )

    gr.Examples(examples=examples, inputs=[init_image, prompt_choice])

demo.launch(share=True)
